In [1]:
import torch
import json
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from multiprocess import Pool
import itertools

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))
    
def new_path(f):
    return f.replace('/wavs/', '/dac/').replace('.wav', '.dac')

def new_path_audio(f):
    return f.replace('_processed/', '_processed_trim/')

with open('/home/nafis/code/text-to-speech/dia/ft-dia/config.json') as fopen:
    config = json.load(fopen)
    
text_length = config['data']['text_length']
audio_length = config['data']['audio_length']
codebook_size = config['data']['channels']

max_text = config['data']['text_length']
pad_tok = config['data']['text_pad_value']
max_audio = config['data']['audio_length']

In [2]:
dataset = load_dataset("parquet", data_files={'train': 'permutate.parquet'})
dataset

DatasetDict({
    train: Dataset({
        features: ['reference_audio', 'reference_text', 'target_audio', 'target_text'],
        num_rows: 42600
    })
})

In [3]:
dataset = load_dataset("parquet", data_files={'train': 'permutate.parquet'})
dataset = dataset["train"]
dataset

Dataset({
    features: ['reference_audio', 'reference_text', 'target_audio', 'target_text'],
    num_rows: 42600
})

In [4]:
def loop(indices):
    indices, _ = indices
    lengths = []
    
    dataset = load_dataset("parquet", data_files={'train': 'permutate.parquet'})
    dataset = dataset["train"]
    for i in tqdm(indices):
        data = dataset[i]
        reference_audio = data['reference_audio'] 
        reference_text = data['reference_text']
        target_audio = data['target_audio']
        target_text = data['target_text']
        text = f'[S1] {reference_text}[S1] {target_text}'
        encoder_l = len(list(text.encode('utf-8')))
        files = [reference_audio, target_audio]
        decoder_l = 0
        for f in files:
            new_f = new_path(f)
            #print(new_f)
            with open(new_f) as fopen:
                d = json.load(fopen)
            d = np.array(d)
            if d.shape[1] != codebook_size:
                d = d.T
            decoder_l += d.shape[0]
    
        lengths.append({
            'i': i,
            'encoder_l': encoder_l,
            'decoder_l': decoder_l
        })
    return lengths

In [5]:
lengths = loop((range(10), 0))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 390.72it/s]


In [6]:
lengths = multiprocessing(range(len(dataset)), loop, cores = 20)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2130/2130 [00:16<00:00, 129.24it/s]


In [7]:
rows = sorted(lengths, key = lambda x: x['decoder_l'])

In [8]:
rows[-1]

{'i': 41890, 'encoder_l': 260, 'decoder_l': 2258}

In [9]:
maxlen = 384
maxlen_encoder = 0
data, temp, l, l_encoder = [], [], 0, 0
for r in tqdm(rows):
    if r['decoder_l'] > maxlen:
        continue
        
    if l + r['decoder_l'] >= maxlen:
        data.append(temp)
        temp = [r['i']]
        maxlen_encoder = max(maxlen_encoder, l_encoder)
        l = r['decoder_l']
        l_encoder = r['encoder_l']
    else:
        l += r['decoder_l']
        l_encoder += r['encoder_l']
        temp.append(r['i'])

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 42600/42600 [00:00<00:00, 1498807.60it/s]


In [10]:
maxlen_encoder

99

In [11]:
len(data)

1902

In [12]:
dataset[data[-1][0]]

{'reference_audio': '/home/nafis/code/text-to-speech/data/data/indspeech_teldialog_lvcsr/wavs/Ind335_M_B_C_appl_0857.wav',
 'reference_text': 'Bisa kelar hari ini juga?',
 'target_audio': '/home/nafis/code/text-to-speech/data/data/indspeech_teldialog_lvcsr/wavs/Ind335_M_B_C_appl_0870.wav',
 'target_text': 'Dua sembilan.'}

In [13]:
import json

with open('merged-dia-384-preprocessing.json', 'w') as fopen:
    json.dump(data, fopen)

In [16]:
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="merged-dia-4096.json",
    path_in_repo="merged-dia-4096.json",
    repo_id="mesolitica/Malaysian-Emilia-Audio-Tokens",
    repo_type="dataset",
)

merged-dia-4096.json:   0%|          | 0.00/38.9M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/mesolitica/Malaysian-Emilia-Audio-Tokens/commit/e9df12dc61fa266982f2856c0ac359ae71963732', commit_message='Upload merged-dia-4096.json with huggingface_hub', commit_description='', oid='e9df12dc61fa266982f2856c0ac359ae71963732', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/mesolitica/Malaysian-Emilia-Audio-Tokens', endpoint='https://huggingface.co', repo_type='dataset', repo_id='mesolitica/Malaysian-Emilia-Audio-Tokens'), pr_revision=None, pr_num=None)

In [ ]:
from datasets import load_dataset, DatasetDict
from huggingface_hub import HfApi, HfFolder

from huggingface_hub import login
login(token)

dataset = load_dataset("parquet", data_files={'train': 'permutate.parquet'})
dataset.push_to_hub("atmatechai/permutate-dataset")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/43 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


CommitInfo(commit_url='https://huggingface.co/datasets/atmatechai/permutate-dataset/commit/22278be67d880520adb8a4dad75e3802fd878081', commit_message='Upload dataset', commit_description='', oid='22278be67d880520adb8a4dad75e3802fd878081', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/atmatechai/permutate-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='atmatechai/permutate-dataset'), pr_revision=None, pr_num=None)